# LegalGPT: train the local LoRA model on a Colab T4 (3B or 7B)

1. **Runtime → Change runtime type → T4 GPU.**
2. Put `legalgpt_colab.zip` (made on the laptop by `backend/scripts/colab/make_package.py`) in
   Google Drive under `MyDrive/legalgpt/`.
3. Set `MODEL_SIZE` below, then **Runtime → Run all**.

Everything is saved to Google Drive as it goes: checkpoints every 20 steps, and the benchmark answers
one by one. If Colab disconnects, reconnect and **Run all** again; training and the benchmark resume
where they stopped.

Rough times on a free T4: 3B about 3.5-4 h of training (2 passes); 7B about 4-5 h (1 pass), likely over
several sessions. Then the benchmark (150 questions) takes about 45-90 min.

In [ ]:
MODEL_SIZE = "3B"          # "3B" (everyday local model) or "7B" (heavy local model)
WITH_JUDGMENTS = False     # True only if the zip was made with --with-judgments

BASE = {"3B": "Qwen/Qwen2.5-3B-Instruct", "7B": "Qwen/Qwen2.5-7B-Instruct"}[MODEL_SIZE]
RUN = f"v0.7-{MODEL_SIZE.lower()}"
DRIVE = "/content/drive/MyDrive/legalgpt"
REPO = "/content/legaltech"
print(BASE, RUN)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, zipfile
if not os.path.exists(REPO):
    zipfile.ZipFile(f"{DRIVE}/legalgpt_colab.zip").extractall(REPO)
# benchmark answers go straight to Drive, so a disconnect loses nothing
os.makedirs(f"{DRIVE}/bench", exist_ok=True)
os.makedirs(f"{REPO}/data/reports/bench", exist_ok=True)
link = f"{REPO}/data/reports/bench/baselines"
if not os.path.exists(link):
    os.symlink(f"{DRIVE}/bench", link)
print(os.listdir(REPO))

In [ ]:
# Colab's own CUDA build of PyTorch is kept; the rest matches the laptop's versions
!pip install -q -r {REPO}/backend/scripts/colab/requirements-colab.txt
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bf16", torch.cuda.is_bf16_supported(),
      "| transformers", transformers.__version__, "| peft", peft.__version__)

## Train
Same recipe as the laptop's v0.7: IPC Q&A, scenario answers, ~30% scenario refusals, and the query
rewrite distilled from larger models. On the T4 it trains in float16 (no bfloat16 on that GPU).

In [ ]:
judg = "" if WITH_JUDGMENTS else "--no-judgments"
EPOCHS = 1 if MODEL_SIZE == "7B" else 2   # 7B: one pass (about half the time on a free T4)
!cd {REPO} && python backend/scripts/train_lora.py --base-model {BASE} --version v0.2 {judg} \
    --ipc-qa --no-case-refusals --scenarios --scenario-refusals 0.3 --rewrites --epochs {EPOCHS} \
    --save-steps 20 --out {DRIVE}/lora/{RUN}

## Benchmark (fully offline: the local model rewrites the question itself)
150 questions, the same sample as every other system in `docs/reports/benchmark_results.md`.

In [ ]:
dtype = "4bit" if MODEL_SIZE == "7B" else "auto"   # a 7B in 16-bit does not fit the T4
!cd {REPO} && ADAPTER_PATH={DRIVE}/lora/{RUN} LOCAL_LLM_DTYPE={dtype} \
    python backend/scripts/run_baselines.py run --rewrite --suffix _{RUN} --systems legalgpt-lora
!cd {REPO} && python backend/scripts/run_baselines.py score

## 7B only: convert the adapter for llama.cpp (the laptop's fast path)
The laptop runs the 7B through llama.cpp with a 4-bit GGUF base model. This converts the LoRA
adapter to GGUF (a few minutes).

In [ ]:
if MODEL_SIZE == "7B":
    if not os.path.exists("/content/llama.cpp"):
        !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !pip install -q /content/llama.cpp/gguf-py sentencepiece
    from huggingface_hub import snapshot_download
    base_cfg = snapshot_download(BASE, allow_patterns=["*.json", "tokenizer*", "*.model"])
    !python /content/llama.cpp/convert_lora_to_gguf.py --base {base_cfg} --outtype f16 \
        --outfile {DRIVE}/lora/{RUN}.gguf {DRIVE}/lora/{RUN}
    print(os.path.getsize(f"{DRIVE}/lora/{RUN}.gguf") / 1e6, "MB")

## Bring it to the laptop
Download from Google Drive (`MyDrive/legalgpt/`):
- **3B:** the folder `lora/v0.7-3b`, without its `_trainer` subfolder. Put it in `data/models/lora/` and
  set `ADAPTER_PATH=data/models/lora/v0.7-3b` in `.env`.
- **7B:** the file `lora/v0.7-7b.gguf`. Put it in `data/models/gguf/`; see
  `backend/scripts/colab/README.md` for the base model file and the `.env` lines.
- **Results:** `bench/legalgpt-lora_v0.7-*.jsonl`, into `data/reports/bench/baselines/`.